# Stage 0 — Conformal Prediction Simulation (Colab)

This notebook tests our medical hallucination-filtering method on **fake data** where we
already know the right answers. No GPU. No datasets. Pure NumPy. Runs in a few seconds.

It checks the three things our paper depends on:

1. **Validity** — the safety promise actually holds on new data.
2. **Efficiency** — a better confidence score keeps more good info (but never affects safety).
3. **The headline** — treating dangerous claims more strictly catches far more dangerous lies,
   at the same overall amount of information shown.

**How to run:** `Runtime → Run all` (or press Shift+Enter on each cell, top to bottom).
You do NOT need a GPU. `Runtime → Change runtime type → CPU` is fine.

## Step 1 — Setup

Colab already has NumPy installed, so there's nothing to download. This cell just imports it
and sets a fixed random seed so your results match every time.

In [ ]:
import numpy as np

# One global random generator. The fixed seed (0) makes results reproducible.
RNG = np.random.default_rng(0)
print("NumPy version:", np.__version__)
print("Setup done.")

## Step 2 — The method (Conformal Risk Control)

This is the actual technique. Two things matter:

- **Keep rule:** keep a claim if its confidence `c` is at least the threshold `lambda`.
- **The selection rule:** pick the *lowest* (most permissive) threshold whose error,
  plus a small safety cushion `1/(n+1)`, still stays under your budget `alpha`.

The cushion is why you need lots of calibration data: with few examples it's huge,
with hundreds it's tiny.

In [ ]:
from dataclasses import dataclass

@dataclass
class Claim:
    text: str
    confidence: float        # c in [0,1], high = likely true
    tier: str = "benign"     # "dangerous" or "benign"
    label: int = -1          # 1 = hallucination, 0 = true (known only in calibration)


def crc_calibrate(confidences, labels, alpha, B=1.0):
    """Find the smallest threshold whose corrected risk stays <= alpha.
    Returns np.inf if nothing works (-> flag everything, the honest fallback)."""
    confidences = np.asarray(confidences, dtype=float)
    labels = np.asarray(labels, dtype=int)
    n = len(confidences)
    if n == 0:
        return np.inf
    for lam in np.sort(np.unique(confidences)):       # try low -> high, stop at first OK
        kept = confidences >= lam
        rhat = np.sum(kept & (labels == 1)) / n       # fraction kept-and-wrong
        corrected = (n / (n + 1)) * rhat + B / (n + 1)
        if corrected <= alpha:
            return float(lam)
    return np.inf


def crc_calibrate_stratified(claims, alphas, B=1.0):
    """One independent threshold per severity tier (the severity-aware version)."""
    thresholds = {}
    for tier, alpha in alphas.items():
        conf = np.array([c.confidence for c in claims if c.tier == tier])
        lab = np.array([c.label for c in claims if c.tier == tier])
        thresholds[tier] = crc_calibrate(conf, lab, alpha, B=B)
    return thresholds


def apply_global(claims, lam):
    return np.array([c.confidence >= lam for c in claims])

def apply_stratified(claims, thresholds):
    return np.array([c.confidence >= thresholds[c.tier] for c in claims])

print("Method loaded.")

## Step 3 — Measurements

Two ways to measure "how much bad stuff got through" — they give different numbers (remember
the snag we discussed):

- **Marginal risk** = bad claims ÷ **all** claims. *This is the one the math guarantees.*
- **FDR risk** = bad claims ÷ **shown** claims. More intuitive, but NOT guaranteed by this method.

We track both so you can see the gap. We also track **retention** = how many *true* claims we kept.

In [ ]:
def realized_risk_marginal(claims, kept):
    """bad kept claims / TOTAL claims  (this is what CRC guarantees)"""
    kept = np.asarray(kept); labels = np.array([c.label for c in claims])
    return np.sum(kept & (labels == 1)) / len(claims) if len(claims) else 0.0

def realized_risk_fdr(claims, kept):
    """bad kept claims / KEPT claims  (intuitive, but NOT guaranteed)"""
    kept = np.asarray(kept); labels = np.array([c.label for c in claims])
    nk = kept.sum()
    return np.sum(kept & (labels == 1)) / nk if nk else 0.0

def retention(claims, kept):
    """true claims we kept / all true claims  (information preserved)"""
    kept = np.asarray(kept); labels = np.array([c.label for c in claims])
    nt = np.sum(labels == 0)
    return np.sum(kept & (labels == 0)) / nt if nt else 0.0

def tier_risk_marginal(claims, kept, tier):
    kept = np.asarray(kept); idx = np.array([c.tier == tier for c in claims])
    labels = np.array([c.label for c in claims]); nt = idx.sum()
    return np.sum(kept & idx & (labels == 1)) / nt if nt else 0.0

def count_kept_dangerous_halluc(claims, kept):
    kept = np.asarray(kept); labels = np.array([c.label for c in claims])
    tiers = np.array([c.tier for c in claims])
    return int(np.sum(kept & (labels == 1) & (tiers == "dangerous")))

def auroc(confidences, labels):
    """How well the score ranks true claims above lies. 0.5 = useless, 1.0 = perfect."""
    confidences = np.asarray(confidences); labels = np.asarray(labels)
    pos = confidences[labels == 0]; neg = confidences[labels == 1]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    order = np.argsort(np.concatenate([pos, neg]))
    ranks = np.empty_like(order, dtype=float); ranks[order] = np.arange(1, len(order)+1)
    return (ranks[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg))

print("Measurements loaded.")

## Step 4 — Fake data generator

We invent claims where we secretly know the truth, so we can check if the method works.
- True claims get **high** confidence, lies get **low** confidence (with overlap, like real life).
- `score_strength` controls how good the score is: 2.0 = good, 0.0 = random.
- Each claim is tagged **dangerous** or **benign** (independent of whether it's true).

In [ ]:
def make_claims(n, halluc_rate=0.30, dangerous_frac=0.40, score_strength=2.0, rng=RNG):
    labels = (rng.random(n) < halluc_rate).astype(int)            # 1 = lie
    latent = rng.normal(loc=np.where(labels == 0, +1.0, -1.0) * score_strength,
                        scale=2.0, size=n)
    confidence = 1.0 / (1.0 + np.exp(-latent))                    # squash into (0,1)
    tiers = np.where(rng.random(n) < dangerous_frac, "dangerous", "benign")
    return [Claim(text=f"claim_{i}", confidence=float(confidence[i]),
                  tier=str(tiers[i]), label=int(labels[i])) for i in range(n)]

def split(claims, frac_cal=0.5, rng=RNG):
    """Split into a calibration half (learn threshold) and a test half (check it)."""
    idx = rng.permutation(len(claims)); k = int(len(claims) * frac_cal)
    return [claims[i] for i in idx[:k]], [claims[i] for i in idx[k:]]

# quick peek
demo = make_claims(5, rng=np.random.default_rng(42))
for c in demo:
    print(f"  {c.tier:9s} conf={c.confidence:.2f}  {'LIE' if c.label else 'true'}")
print("\nGenerator ready.")

## Experiment 1 — Does the safety promise actually hold?

We learn a threshold on the calibration half, apply it to the unseen test half, and measure
the error. We repeat 500 times with different random splits. The promise is "in expectation"
(on average), so we check the **average** marginal risk stays under the budget.

In [ ]:
def experiment_validity(alpha=0.10, n_total=2000, n_trials=500):
    print(f"Target budget alpha = {alpha}\n")
    pool = make_claims(n_total)
    print(f"score AUROC on full pool: {auroc([c.confidence for c in pool],[c.label for c in pool]):.3f}")
    marg, fdr, rets = [], [], []
    for t in range(n_trials):
        rng = np.random.default_rng(1000 + t)
        cal, test = split(pool, rng=rng)
        lam = crc_calibrate([c.confidence for c in cal], [c.label for c in cal], alpha)
        if not np.isfinite(lam):
            continue
        kept = apply_global(test, lam)
        marg.append(realized_risk_marginal(test, kept))
        fdr.append(realized_risk_fdr(test, kept))
        rets.append(retention(test, kept))
    marg = np.array(marg)
    print(f"\nMEAN marginal risk (GUARANTEED): {marg.mean():.4f}   ->  "
          f"{'VALID (<= alpha)' if marg.mean() <= alpha else 'INVALID'}")
    print(f"MEAN fdr-style risk (not guaranteed): {np.mean(fdr):.4f}")
    print(f"MEAN retention of true claims:        {np.mean(rets):.4f}")

experiment_validity()

**What you should see:** marginal risk ≈ 0.099, just under the 0.10 budget → the guarantee is real.
The fdr risk (≈ 0.13) sits above 0.10 — that's the "snag", shown for honesty.

## Experiment 2 — Does a better confidence score help?

We run the same method with a good score, a weak score, and a totally random score.
Watch: safety holds in all three, but a worse score forces us to throw away more good info.

In [ ]:
def experiment_efficiency(alpha=0.10, n_total=2000, n_trials=300):
    for name, strength in [("good score", 2.0), ("weak score", 0.6), ("random score", 0.0)]:
        pool = make_claims(n_total, score_strength=strength, rng=np.random.default_rng(7))
        aur = auroc([c.confidence for c in pool], [c.label for c in pool])
        rets, marg = [], []
        for t in range(n_trials):
            rng = np.random.default_rng(2000 + t)
            cal, test = split(pool, rng=rng)
            lam = crc_calibrate([c.confidence for c in cal], [c.label for c in cal], alpha)
            if not np.isfinite(lam):
                rets.append(0.0); marg.append(0.0); continue
            kept = apply_global(test, lam)
            rets.append(retention(test, kept)); marg.append(realized_risk_marginal(test, kept))
        print(f"{name:13s} AUROC={aur:.3f} | marginal risk={np.mean(marg):.3f} (valid) "
              f"| retention={np.mean(rets):.3f}")

experiment_efficiency()

**What you should see:** risk stays ≈ 0.10 every time (safety never breaks), but retention
falls hard (≈ 0.94 → 0.57 → 0.34) as the score gets worse. **Score quality buys information,
not safety.** This is the most important idea to internalize before building the real score.

## Experiment 3 — The headline result

Compare two systems at the **same overall retention** (we auto-tune the global system's budget
so the comparison is fair):
- **Global:** one threshold for everything.
- **Tiered (ours):** strict budget on dangerous claims, lenient on benign ones.

The win: the tiered system shows far fewer **dangerous** lies for the same total information.

In [ ]:
def experiment_severity(n_total=4000, n_trials=300):
    alpha_dang, alpha_ben = 0.05, 0.20
    pool = make_claims(n_total, halluc_rate=0.30, dangerous_frac=0.40,
                      score_strength=2.0, rng=np.random.default_rng(3))

    def run_tiered(cal, test):
        thr = crc_calibrate_stratified(cal, {"dangerous": alpha_dang, "benign": alpha_ben})
        if not all(np.isfinite(v) for v in thr.values()): return None
        return apply_stratified(test, thr), thr

    def run_global(cal, test, ag):
        lam = crc_calibrate([c.confidence for c in cal], [c.label for c in cal], ag)
        if not np.isfinite(lam): return None
        return apply_global(test, lam), lam

    # target retention from the tiered system
    tr = []
    for t in range(n_trials):
        cal, test = split(pool, rng=np.random.default_rng(4000 + t))
        out = run_tiered(cal, test)
        if out: tr.append(retention(test, out[0]))
    target = np.mean(tr)

    # tune global budget to match that retention (fair comparison)
    best_ag, best_gap = None, 1e9
    for ag in np.arange(0.02, 0.41, 0.01):
        rr = []
        for t in range(60):
            cal, test = split(pool, rng=np.random.default_rng(4000 + t))
            out = run_global(cal, test, ag)
            if out: rr.append(retention(test, out[0]))
        if abs(np.mean(rr) - target) < best_gap:
            best_gap, best_ag = abs(np.mean(rr) - target), ag

    g_ret, g_dk, g_dm = [], [], []
    t_ret, t_dk, t_dm = [], [], []
    ex = None
    for t in range(n_trials):
        cal, test = split(pool, rng=np.random.default_rng(5000 + t))
        gt, tt = run_global(cal, test, best_ag), run_tiered(cal, test)
        if gt is None or tt is None: continue
        gkept, glam = gt; tkept, thr = tt; ex = (glam, thr)
        g_ret.append(retention(test, gkept)); g_dk.append(count_kept_dangerous_halluc(test, gkept))
        g_dm.append(tier_risk_marginal(test, gkept, "dangerous"))
        t_ret.append(retention(test, tkept)); t_dk.append(count_kept_dangerous_halluc(test, tkept))
        t_dm.append(tier_risk_marginal(test, tkept, "dangerous"))

    glam, thr = ex
    print(f"tiered budgets: dangerous={alpha_dang}, benign={alpha_ben}")
    print(f"matched global budget (auto-tuned): {best_ag:.2f}")
    print(f"thresholds -> global={glam:.3f} | dangerous={thr['dangerous']:.3f}, benign={thr['benign']:.3f}")
    print("-" * 60)
    print(f"{'':24s}{'GLOBAL':>10s}{'TIERED':>10s}")
    print(f"{'overall retention':24s}{np.mean(g_ret):>10.3f}{np.mean(t_ret):>10.3f}   (matched)")
    print(f"{'dangerous risk':24s}{np.mean(g_dm):>10.3f}{np.mean(t_dm):>10.3f}")
    print(f"{'dangerous lies shown':24s}{np.mean(g_dk):>10.1f}{np.mean(t_dk):>10.1f}")
    print("-" * 60)
    red = 100 * (1 - np.mean(t_dk) / np.mean(g_dk))
    print(f"=> at the same overall retention, the tiered system shows {red:.0f}% fewer dangerous lies.")

experiment_severity()

**What you should see:** matched retention (~0.93 both), but the tiered system's dangerous
risk drops to ≈ 0.05 (meeting its strict budget) and it shows ~50% fewer dangerous lies.
**That single contrast is your paper.**

---

## You're done with Stage 0 ✅

You've verified, on a computer, that the method is valid, that score quality controls information
(not safety), and that the severity-aware idea works. Next stages (real medical data, a real model)
need a GPU — we'll tackle those one at a time later.